# Jageocoder住所→座標変換 PoC（実験用Notebook）

このNotebookは、**Jageocoderによる住所→座標変換が現在実用可能かどうかを確認するための実験用Notebook**です。

**sheltermatch本体（`sheltermatch.ipynb`）ではありません。** ここでの結果をsheltermatch本体へ統合する処理は、このNotebookには含まれていません。

## 目的

以前、sheltermatchでの住所→座標変換の検討では、以下の問題が確認されていました。

- Google Colab側のPythonバージョンとの互換性
- Jageocoder / jageocoder-converter間のAPI互換性
- 辞書生成時の外部データ取得失敗
- 配布元でのHTTP 403等

Jageocoderは継続的に更新されているため、このNotebookでは現時点のバージョンで以下をゼロベースで再確認します。

1. Colabで現在のJageocoderを正常にインストール・importできるか
2. 沖縄県だけの辞書を生成できるか
3. その辞書をJageocoderから利用できるか
4. 糸満市の公開住所を住所→緯度経度へ変換できるか
5. 一致した住所と一致レベルを確認できるか

## 進め方

上から順にすべてのセルを実行してください（「ランタイム → すべてのセルを実行」）。

外部データの取得に失敗した場合、このNotebookは失敗を握りつぶさず、失敗した段階・取得先・エラー概要を表示して停止します。**別のジオコーダーへの自動切替や、非公式ミラーの利用は行いません。** 失敗そのものも、今回の検証結果として意味があります。

## このNotebookが行わないこと

- sheltermatch本体への統合
- ABR Geocoder等、他のジオコーダーへの自動フォールバック
- Google Maps等の外部Geocoding APIの利用
- 全国辞書の作成
- 個人の住所を使ったテスト


In [ ]:
# ===== 設定 =====
# このPoCで変更が必要な項目はこれだけです。

# 辞書を作成する対象の都道府県コード（JIS X 0401、2桁）。沖縄県 = "47"。
PREF_CODE = "47"

# 変換テストの対象とする市区町村名（表示・確認用。辞書作成自体は都道府県単位で行う）。
TARGET_CITY = "糸満市"

# 生成した辞書の保存先ディレクトリ（Colabランタイム上の一時ディレクトリ。ランタイムが
# リセットされると消える。sheltermatch本体のようにアップロードで受け取るデータではなく、
# このNotebook内で生成・利用するデータのため、ここに保存先を置く）。
DICTIONARY_DIR = "./jageocoder_okinawa_dict"

print("設定を読み込みました。")
print(f"  PREF_CODE      = '{PREF_CODE}'")
print(f"  TARGET_CITY    = '{TARGET_CITY}'")
print(f"  DICTIONARY_DIR = '{DICTIONARY_DIR}'")


In [ ]:
# ===== 環境確認・パッケージ導入 =====
import io
import logging
import platform
import re
import sys

import pandas as pd
from google.colab import files

# 以降の各ステップの結果を記録する（最後の「検証結果サマリ」セルで使用する）。
poc_status = {
    "python_env": None,
    "jageocoder_import": None,
    "okinawa_dictionary": None,
    "geocoder_init": None,
    "address_search": None,
    "csv_batch": None,
}

print(f"Python version : {sys.version}")
print(f"platform       : {platform.platform()}")
poc_status["python_env"] = True

# 実機検証の結果、2026年9月時点でjageocoder(最新2.2系)とjageocoder-converter(最新2.0.3)を
# バージョン指定なしで同時にインストールすると、jageocoder-converter 2.0.3はjageocoder==2.1.0を
# 要求するため、pipはjageocoder-converterを古い1.2.0系へ引き下げてしまう。しかし
# jageocoder-converter 1.2.0はjageocoder 2.2系の内部実装変更（SQLAlchemyベースのtree.engine
# 廃止）に対応しておらず、辞書生成が 'LocalTree' object has no attribute 'engine' で失敗する
# ことを実機確認した。そのため、jageocoder-converterが要求するjageocoder==2.1.0を明示的に
# 固定する（理由のない固定ではなく、この組合せでのみ辞書生成の実行まで進むことを確認したための
# 固定。jageocoder自体の最新版ではない点に注意）。
# また、jageocoder-converter 2.0.3はコード内でsqlalchemyを使用するが、パッケージの依存関係
# には含まれていないため、別途明示的にインストールする（これも実機確認した現在の挙動）。
%pip install -q "jageocoder==2.1.0" "jageocoder-converter==2.0.3" sqlalchemy

try:
    import jageocoder
    import jageocoder_converter

    print(f"jageocoder version           : {jageocoder.__version__}")
    print(f"jageocoder-converter version : {jageocoder_converter.__version__}")
    poc_status["jageocoder_import"] = True
except Exception as e:
    poc_status["jageocoder_import"] = False
    print("Jageocoder / jageocoder-converterのimportに失敗しました。")
    print(f"エラー概要: {type(e).__name__}: {e}")


In [ ]:
# ===== 沖縄県限定辞書の準備 =====
# jageocoder_converterが提供する公式のPython API（jageocoder_converter.convert()。
# `python -m jageocoder_converter` CLIが内部で呼び出しているものと同じ関数）を直接呼び出し、
# PREF_CODEで指定した都道府県だけを含む辞書を生成する（全国辞書は作成しない）。
# 進行状況はjageocoder_converterが出力するINFOログのみを表示し、レコード単位の詳細ログ
# （DEBUG）は表示しない。

_converter_logger = logging.getLogger("jageocoder_converter")
_converter_logger.setLevel(logging.INFO)
if not _converter_logger.handlers:
    _handler = logging.StreamHandler()
    _handler.setFormatter(logging.Formatter("[辞書生成] %(message)s"))
    _converter_logger.addHandler(_handler)


def _find_download_url(exc):
    """例外のトレースバックから、実際にアクセスしようとしていたURL（ローカル変数'url'）を
    探して返す。見つからない場合はNoneを返す（推測でURLを作らない）。"""
    tb = exc.__traceback__
    while tb is not None:
        value = tb.tb_frame.f_locals.get("url")
        if isinstance(value, str) and value.startswith("http"):
            return value
        tb = tb.tb_next
    return None


if not poc_status.get("jageocoder_import"):
    print("Jageocoder / jageocoder-converterのimportに失敗しているため、このセルはスキップします。")
    poc_status["okinawa_dictionary"] = False
else:
    print(
        "quiet=Trueのため、各データ配布元の利用規約への同意プロンプトは表示されません。"
        "実行前に、国土数値情報・国土地理院・アドレス・ベース・レジストリ等、"
        "jageocoder_converterが参照する各データ配布元の利用規約を確認してください。"
    )
    try:
        jageocoder_converter.convert(prefs=[PREF_CODE], db_dir=DICTIONARY_DIR, quiet=True)
        print(f"'{DICTIONARY_DIR}' へ都道府県コード{PREF_CODE}のみの辞書を生成しました。")
        poc_status["okinawa_dictionary"] = True
    except Exception as e:
        poc_status["okinawa_dictionary"] = False
        found_url = _find_download_url(e)
        status_match = re.search(r"\b([1-5]\d{2})\b", str(e))
        print("Jageocoder本体: 成功")
        print("沖縄県辞書生成: 失敗")
        print("失敗段階: 沖縄県限定辞書の生成（jageocoder_converter.convert）")
        print(f"取得先: {found_url or '特定できず（jageocoder_converterが参照する公式データ配布元のいずれか）'}")
        print(f"HTTP status等: {status_match.group(1) if status_match else '不明（下記エラー概要を参照）'}")
        print(f"エラー概要: {type(e).__name__}: {e}")


In [ ]:
# ===== Jageocoder初期化 =====
# 生成した沖縄県限定辞書を明示的に指定してJageocoderを初期化する。

if not poc_status.get("okinawa_dictionary"):
    print("沖縄県限定辞書の生成に失敗しているため、このセルはスキップします。")
    poc_status["geocoder_init"] = False
else:
    try:
        jageocoder.init(db_dir=DICTIONARY_DIR)
        print(f"辞書を初期化しました（db_dir='{DICTIONARY_DIR}'）。")
        print(f"初期化状態: {jageocoder.is_initialized()}")
        print(f"辞書ディレクトリ: {jageocoder.get_db_dir()}")
        print(f"辞書バージョン: {jageocoder.installed_dictionary_version(db_dir=DICTIONARY_DIR)}")
        poc_status["geocoder_init"] = True
    except Exception as e:
        poc_status["geocoder_init"] = False
        print("Jageocoderの初期化に失敗しました。")
        print(f"エラー概要: {type(e).__name__}: {e}")


In [ ]:
# ===== 公開住所による変換テスト =====
# 個人情報は使用しない。糸満市内の公開されている公共施設の住所を使用する
# （リポジトリ内の要支援者テストデータ test/residents_sample_enriched.csv は架空住所のため、
# ここでは使用しない）。実際の住所と異なる場合は、下記TEST_ADDRESSESを書き換えて構わない。

TEST_ADDRESSES = [
    "沖縄県糸満市潮崎町1丁目1番地",  # 糸満市役所
    "沖縄県糸満市字糸満673",  # 糸満市立糸満小学校
    "沖縄県糸満市真栄里1448番地",  # 糸満市立中央図書館
]

geocode_results = []

if not poc_status.get("geocoder_init"):
    print("Jageocoderが初期化されていないため、このセルはスキップします。")
    poc_status["address_search"] = False
else:
    for address in TEST_ADDRESSES:
        try:
            result = jageocoder.search(address)
            candidates = result.get("candidates", [])
        except Exception as e:
            geocode_results.append(
                {
                    "input_address": address,
                    "matched_address": None,
                    "latitude": None,
                    "longitude": None,
                    "match_level": None,
                    "status": f"error: {type(e).__name__}: {e}",
                }
            )
            continue

        if candidates:
            best = candidates[0]
            geocode_results.append(
                {
                    "input_address": address,
                    "matched_address": "".join(best.get("fullname", [])),
                    "latitude": best.get("y"),
                    "longitude": best.get("x"),
                    "match_level": best.get("level"),
                    "status": "matched",
                }
            )
        else:
            geocode_results.append(
                {
                    "input_address": address,
                    "matched_address": None,
                    "latitude": None,
                    "longitude": None,
                    "match_level": None,
                    "status": "no_match",
                }
            )

    matched_count = sum(1 for r in geocode_results if r["status"] == "matched")
    print(f"{len(TEST_ADDRESSES)}件中{matched_count}件を変換しました。")
    poc_status["address_search"] = matched_count > 0


In [ ]:
# ===== 結果確認 =====
# Jageocoderのsearch()が実際に返す値（matched/candidates. 各候補はid/name/x/y/level/priority/
# note/fullname）だけを表示する。Jageocoderが返していない精度を独自に推測しない。
# match_levelの説明文は、Jageocoder自身のAddressLevel定数（1~8の意味はjageocoder.address.
# AddressLevelのdocstringに定義されている）をそのまま使う。

if not geocode_results:
    print("変換テストの結果がありません。")
else:
    from jageocoder.address import AddressLevel

    # AddressLevelのdocstringに定義された1~8の意味そのもの（独自の精度解釈は加えない）。
    LEVEL_LABELS = {
        AddressLevel.PREF: "都道府県",
        AddressLevel.COUNTY: "郡・支庁・振興局",
        AddressLevel.CITY: "市町村および特別区",
        AddressLevel.WARD: "政令市の区",
        AddressLevel.OAZA: "大字",
        AddressLevel.AZA: "字",
        AddressLevel.BLOCK: "地番または住居表示実施地域の街区",
        AddressLevel.BLD: "枝番または住居表示実施地域の住居番号",
    }

    results_df = pd.DataFrame(geocode_results)
    results_df["match_level_label"] = results_df["match_level"].apply(
        lambda level: LEVEL_LABELS.get(int(level)) if pd.notna(level) else None
    )
    display(
        results_df[
            [
                "input_address",
                "matched_address",
                "latitude",
                "longitude",
                "match_level",
                "match_level_label",
                "status",
            ]
        ]
    )


In [ ]:
# ===== CSVアップロードによる簡易確認（検証用） =====
# 任意のセル。id,address列を持つCSVをアップロードすると、一括で住所→座標変換を試せる。
# あくまでPoCの成立確認用であり、sheltermatch本体のCSV仕様との統合はここでは行わない。
# 変換できなかった行も削除しない。アップロードをキャンセルした場合はこのセルをスキップする。

if not poc_status.get("geocoder_init"):
    print("Jageocoderが初期化されていないため、このセルはスキップします。")
    poc_status["csv_batch"] = None
else:
    print("id,address列を持つCSVを選択してください（試さない場合はアップロードをキャンセルしてください）。")
    uploaded_csv = files.upload()

    if not uploaded_csv:
        print("CSVがアップロードされなかったため、このセルはスキップされました。")
        poc_status["csv_batch"] = None
    else:
        csv_filename = list(uploaded_csv.keys())[0]
        addresses_df = pd.read_csv(io.BytesIO(uploaded_csv[csv_filename]))

        if "address" not in addresses_df.columns:
            print("'address'列が見つかりません。id,addressの形式で用意してください。")
            poc_status["csv_batch"] = False
        else:
            matched_addresses, latitudes, longitudes = [], [], []
            match_levels, geocode_statuses = [], []

            for address in addresses_df["address"]:
                try:
                    result = jageocoder.search(str(address))
                    candidates = result.get("candidates", [])
                except Exception as e:
                    matched_addresses.append(None)
                    latitudes.append(None)
                    longitudes.append(None)
                    match_levels.append(None)
                    geocode_statuses.append(f"error: {type(e).__name__}: {e}")
                    continue

                if candidates:
                    best = candidates[0]
                    matched_addresses.append("".join(best.get("fullname", [])))
                    latitudes.append(best.get("y"))
                    longitudes.append(best.get("x"))
                    match_levels.append(best.get("level"))
                    geocode_statuses.append("matched")
                else:
                    matched_addresses.append(None)
                    latitudes.append(None)
                    longitudes.append(None)
                    match_levels.append(None)
                    geocode_statuses.append("no_match")

            addresses_df["matched_address"] = matched_addresses
            addresses_df["latitude"] = latitudes
            addresses_df["longitude"] = longitudes
            addresses_df["match_level"] = match_levels
            addresses_df["geocode_status"] = geocode_statuses

            matched_count = int((addresses_df["geocode_status"] == "matched").sum())
            print(f"{len(addresses_df)}行中{matched_count}行を変換しました。")
            display(addresses_df.head())
            poc_status["csv_batch"] = matched_count > 0


In [ ]:
# ===== 検証結果サマリ =====


def _fmt(value):
    if value is True:
        return "OK"
    if value is False:
        return "NG"
    return "未実施"


print("Jageocoder方式PoC 検証結果サマリ")
print(f"  Python環境        : {_fmt(poc_status.get('python_env'))}")
print(f"  Jageocoder import : {_fmt(poc_status.get('jageocoder_import'))}")
print(f"  沖縄県限定辞書    : {_fmt(poc_status.get('okinawa_dictionary'))}")
print(f"  住所検索          : {_fmt(poc_status.get('address_search'))}")
print(f"  CSV一括変換       : {_fmt(poc_status.get('csv_batch'))}")
print()

if poc_status.get("okinawa_dictionary") and poc_status.get("address_search"):
    print("辞書生成・住所検索まで成立しています。Jageocoder方式を次の検証へ進められる材料があります。")
elif poc_status.get("jageocoder_import") and not poc_status.get("okinawa_dictionary"):
    print(
        "Jageocoder本体は動作しますが、辞書生成段階で成立しませんでした。"
        "上の「沖縄県限定辞書の準備」セルの出力（失敗段階・取得先・エラー概要）を確認してください。"
    )
else:
    print("Jageocoder本体の導入自体が成立しませんでした。上の「環境確認・パッケージ導入」セルの出力を確認してください。")

print()
print("この結果は本Notebook内の検証にとどまり、sheltermatch本体へは反映していません。")
